In [ ]:
import pandas as pd

# 1. Read the original data
df = pd.read_csv(
    "public_jobs_2020_2026.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

print("Original number of records:", len(df))


for col in df.columns:
    if col != "job_description":
        df[col] = df[col].str.strip()


df["career_category"] = df["career_category"].str.lower().replace({
    "data scientist": "Data Scientist",
    "data analyst": "Data Analyst",
    "ml engineer": "ML Engineer",
    "machine learning engineer": "ML Engineer",
    "data engineer": "Data Engineer",
    "ai engineer": "AI Engineer",
    "artificial intelligence engineer": "AI Engineer",
    "nlp engineer": "NLP Engineer",
    "natural language processing engineer": "NLP Engineer"
})


df = df.loc[df["career_category"] != "NLP Engineer"].copy()


for col in df.columns:
    df.loc[
        df[col].str.strip().str.lower().isin(
            ["", "n/a", "na", "null", "none", "nan"]
        ),
        col
    ] = pd.NA

print("Missing values before removal:")
display(df.isna().sum().to_frame("missing_count"))


df = df.dropna().copy()


df = df.drop_duplicates()
df = df.drop_duplicates(subset="job_id", keep="first")


for col in ["date_posted", "collected_at"]:
    df[col] = pd.to_datetime(
        df[col].str.replace("/", "-", regex=False),
        format="%Y-%m-%d",
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")


df = df.dropna().copy()


df["year"] = pd.to_datetime(
    df["date_posted"],
    format="%Y-%m-%d"
).dt.year.astype("Int64")


us_mask = df["location"].str.contains(
    r"\b(?:United States|USA|US)\b|\bU\.S\."
    r"|\b(?:San Francisco|San Mateo|Mountain View|New York|NYC|"
    r"Boston|Seattle|San Jose|Sunnyvale|Bellevue|Pittsburgh|"
    r"Menlo Park|Redwood City|Foster City|Denver|Honolulu|"
    r"Tempe|Palo Alto|Austin|Santa Ana|Costa Mesa|Draper|"
    r"Santa Clara|Livingston|Plano|Atlanta|McCarran|Irvine|"
    r"Los Angeles|Chicago|Long Beach|Raleigh|Dallas|"
    r"Broomfield|Waltham|Nashville|Cottonwood Heights|"
    r"SF Bay Area)\b"
    r"|\b(?:California|Massachusetts|Washington|Illinois|"
    r"Ohio|Texas|Utah|Colorado|Maryland|Virginia|Tennessee)\b",
    case=False,
    na=False,
    regex=True
)


non_us_mask = df["location"].str.contains(
    r"\b(?:Canada|Canadian|Toronto|Ontario|Vancouver|"
    r"United Kingdom|UK|London|Europe|"
    r"Australia|Sydney|Melbourne|Brisbane|"
    r"India|Singapore|Germany|France|Ireland|"
    r"New Zealand|Netherlands|Poland|Israel|"
    r"Japan|China|Brazil|Mexico|APAC|LATAM|"
    r"Worldwide|Global|International)\b",
    case=False,
    na=False,
    regex=True
)

print("Mixed-country records excluded:", (us_mask & non_us_mask).sum())


df = df.loc[us_mask & ~non_us_mask].reset_index(drop=True)


print("Number of records after cleaning:", len(df))
print("Number of columns:", len(df.columns))
print("Remaining missing values:", df.isna().sum().sum())

display(df["career_category"].value_counts())
display(df.head())


with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None
):
    display(
        df["location"]
        .value_counts(dropna=False)
        .rename_axis("location")
        .reset_index(name="job_count")
    )

# 13. Save the cleaned data
df.to_csv(
    "job_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned US job data saved as job_cleaned.csv")

Original number of records: 1083
Missing values before removal:


,missing_count
job_id,0
career_category,0
job_title,0
company,0
job_description,0
date_posted,0
year,0
date_basis,0
location,7
source,0


Mixed-country records excluded: 7
Number of records after cleaning: 688
Number of columns: 12
Remaining missing values: 0


career_category
AI Engineer       275
ML Engineer       202
Data Scientist    127
Data Engineer      57
Data Analyst       27
Name: count, dtype: int64

,job_id,career_category,job_title,company,job_description,date_posted,year,date_basis,location,source,source_url,collected_at
0,Greenhouse:5507088002,Data Scientist,Staff Data Scientist,Databricks,"P-57\nAt Databricks, we are obsessed with enab...",2021-08-27,2021,source_publication_date,"San Francisco, California",Greenhouse,https://databricks.com/company/careers/open-po...,2026-10-01
1,Greenhouse:4006735005,ML Engineer,"Machine Learning Engineer, Search Quality",Glean,About Glean:\nGlean is the Work AI platform th...,2022-04-08,2022,source_publication_date,"Mountain View, CA",Greenhouse,https://job-boards.greenhouse.io/gleanwork/job...,2026-10-01
2,Greenhouse:6322654002,Data Engineer,"Sr. Product Manager, Data Engineering",Databricks,"P-8\nAt Databricks, we are passionate about en...",2022-08-11,2022,source_publication_date,"San Francisco, California",Greenhouse,https://databricks.com/company/careers/open-po...,2026-10-01
3,Greenhouse:6485460002,ML Engineer,Machine Learning Research Engineer,Jane Street,We are looking for a Research Engineer with ro...,2022-11-18,2022,source_publication_date,"New York, New York, United States",Greenhouse,https://www.janestreet.com/join-jane-street/ap...,2026-10-01
4,Greenhouse:4738660,Data Engineer,"Senior Software Engineer, Data Engineering",Robinhood,Join us in building the future of finance.\nOu...,2022-12-08,2022,source_publication_date,"Menlo Park, CA",Greenhouse,https://boards.greenhouse.io/robinhood/jobs/47...,2026-10-01


,location,job_count
0,San Francisco,112
1,Remote - United States,31
2,"San Francisco, CA",30
3,"San Mateo, CA, United States",27
4,"New York, NY",25
5,"Mountain View, California (HQ)",20
6,United States,19
7,US-CA-Menlo Park,17
8,Remote - USA,17
9,"Boston, MA",16


Cleaned US job data saved as job_cleaned.csv
